# Paper III Supplementary Attack Validation Pilot
Non-DP methodological validation of outcome-matched membership attacks and independent reference-loss gap. Not a reproduction of original federated or differentially private models.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
SEED=17286
rng=np.random.default_rng(SEED)
url='https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
try:
    df=pd.read_excel(url,header=1)
except Exception:
    from google.colab import files
    print('UCI download unavailable: upload original default of credit card clients.xlsx')
    uploaded=files.upload()
    df=pd.read_excel(next(iter(uploaded)),header=1)
df.columns=[str(c).strip() for c in df.columns]
target='default payment next month'
assert target in df.columns,df.columns.tolist()
X=df.drop(columns=[target,'ID'],errors='ignore').apply(pd.to_numeric)
y=df[target].astype(int).to_numpy()
assert len(y)==30000
idx=np.arange(len(y))
pool,test=train_test_split(idx,test_size=.2,stratify=y,random_state=SEED)
train,hold=train_test_split(pool,test_size=.25,stratify=y[pool],random_state=SEED+1)
tidx,ridx=train_test_split(train,test_size=.5,stratify=y[train],random_state=SEED+2)
def fit(ids):
    return make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED)).fit(X.iloc[ids],y[ids])
target_model=fit(tidx); reference_model=fit(ridx)
print('Target model test AUC:',roc_auc_score(y[test],target_model.predict_proba(X.iloc[test])[:,1]))
members=[];nonmembers=[]
for cls in (0,1):
    m=tidx[y[tidx]==cls];n=hold[y[hold]==cls];k=min(len(m),len(n))
    members.extend(rng.choice(m,k,replace=False));nonmembers.extend(rng.choice(n,k,replace=False))
members=np.array(members);nonmembers=np.array(nonmembers)
candidate=np.r_[members,nonmembers]
labels=np.r_[np.ones(len(members),int),np.zeros(len(nonmembers),int)]
assert np.array_equal(np.bincount(y[members]),np.bincount(y[nonmembers]))
def loss(model):
    p=np.clip(model.predict_proba(X.iloc[candidate])[:,1],1e-8,1-1e-8)
    return -(y[candidate]*np.log(p)+(1-y[candidate])*np.log1p(-p))
target_loss=loss(target_model);ref_loss=loss(reference_model)
scores={'target_loss':-target_loss,'independent_reference_loss_gap':ref_loss-target_loss}
def metrics(score,truth):
    auc=roc_auc_score(truth,score);fpr,tpr,_=roc_curve(truth,score)
    return float(auc),float(max(tpr[fpr<=.01]))
results=[]
for name,score in scores.items():
    estimates=[]
    for _ in range(200):
        ix=np.r_[rng.choice(np.flatnonzero(labels==1),sum(labels==1),replace=True),
                 rng.choice(np.flatnonzero(labels==0),sum(labels==0),replace=True)]
        estimates.append(metrics(score[ix],labels[ix]))
    arr=np.array(estimates);auc,tpr=metrics(score,labels)
    results.append(dict(attack=name,auc=auc,auc_ci_low=np.percentile(arr[:,0],2.5),auc_ci_high=np.percentile(arr[:,0],97.5),tpr_at_1pct_fpr=tpr,tpr_ci_low=np.percentile(arr[:,1],2.5),tpr_ci_high=np.percentile(arr[:,1],97.5),n_members=len(members),n_nonmembers=len(nonmembers),seed=SEED,bootstrap=200,dp=False))
out=pd.DataFrame(results);print(out.to_string(index=False));out.to_csv('/content/paper3_attack_pilot_metrics.csv',index=False)
print('Non-DP methodological validation pilot ONLY. Not evidence about original federated DP models.')